In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [1]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

/home/robbie/lca-lc-foundations/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [7]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="ollama:qwen2.5:7b",
    tools=tools,
    system_prompt=prompt
)

In [9]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [10]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='5f45054f-da3f-4fd3-bd3d-41a29998a926'),
              AIMessage(content='I will search for information on the `langchain-mcp-adapters` library to provide you with details.\n\n', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-06-04T14:49:09.32330582Z', 'done': True, 'done_reason': 'stop', 'total_duration': 869399491, 'load_duration': 59740125, 'prompt_eval_count': 274, 'prompt_eval_duration': 84536203, 'eval_count': 48, 'eval_duration': 668980515, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019e931c-0863-7570-81dc-5acee96ef845-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '78ddf09c-9319-4caa-b606-53b9e4dfb78b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 274, 'output_tokens': 48, 'total_toke

## Online MCP

In [16]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [17]:
agent = create_agent(
    model="ollama:qwen2.5:7b",
    tools=tools,
)

In [18]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='8851e18e-276a-45c7-be80-bacbdfbcea1a'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-06-04T15:21:13.68964611Z', 'done': True, 'done_reason': 'stop', 'total_duration': 492506468, 'load_duration': 52435520, 'prompt_eval_count': 351, 'prompt_eval_duration': 66249942, 'eval_count': 24, 'eval_duration': 349105216, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019e9339-66ea-7c82-a109-1d7cff20da86-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': '53ad8dc5-b6fe-4b4d-97fe-f060a54693d0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 351, 'output_tokens': 24, 'total_tokens': 375}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "timezone": "America/New_York",\n  "datetime": "2026-06-04T11